# 🧠 内存管理（Part 2：Memory）

**欢迎来到 Kaggle 5 天 Agents 课程 Day 3！**

在上一份 Notebook 中，你学习了 **Sessions** 如何管理对话线程。现在我们加入 **Memory**：一个可检索的长期知识存储，可跨多次会话持续存在。

### 什么是 Memory ❓

Memory 是为 Agent 提供长期知识存储的服务。关键区别：

 > **Session = 短期记忆**（单次对话）
 > 
 > **Memory = 长期知识**（跨多次对话）

用软件工程类比：**Session** 像应用状态（临时），**Memory** 像数据库（持久）。

### 🤔 为什么需要 Memory？

Memory 提供了 Session 单独无法实现的能力：

| 能力 | 含义 | 示例 |
|------------|---------------|---------|
| **跨会话回忆** | 可访问任意历史会话信息 | “这个用户在所有聊天里提过哪些偏好？” |
| **智能抽取** | 由 LLM 整理并提取关键事实 | 存“对花生过敏”，而不是 50 条原始消息 |
| **语义检索** | 按语义匹配，而不只是关键词 | 查“偏好色调”也能命中“最喜欢蓝色” |
| **持久存储** | 应用重启后仍保留 | 让知识随时间累积 |

**示例：** 想象你在与个人助理交流：
- 🗣️ **Session**：它记得你在“当前对话”10 分钟前说的话
- 🧠 **Memory**：它记得你“上周对话”里提到的偏好

### 🎯 你将学到：

- ✅ 初始化 MemoryService 并接入 Agent
- ✅ 将 session 数据写入 memory 存储
- ✅ 搜索并检索 memories
- ✅ 自动化 memory 存储与检索
- ✅ 理解 memory consolidation（概念层面）

#### 📝 实现说明

> 本 Notebook 使用 `InMemoryMemoryService` 进行学习：仅做关键词匹配，不会持久化。
>
> 生产应用建议使用 **Vertex AI Memory Bank**（Day 5 讲解），它支持基于 LLM 的 consolidation 与语义检索，并提供持久化云存储。

---
## ⚙️ 第 1 部分：环境准备

### 1.1：安装依赖

Kaggle Notebooks 环境已预装 Python 版 [google-adk](https://google.github.io/adk-docs/) 及其依赖，因此本 Notebook 无需额外安装。

如果你要在课程外、自己的 Python 环境中安装并使用 ADK，可运行：

```
pip install google-adk
```

### 1.2：配置 Gemini API Key

本 Notebook 使用 [Gemini API](https://ai.google.dev/gemini-api/docs)，需要先完成鉴权。

**1. 获取 API key**

如果你还没有，可在 [Google AI Studio 创建 API key](https://aistudio.google.com/app/api-keys)。

**2. 将 key 添加到 Kaggle Secrets**

接下来，把 API key 作为 Kaggle User Secret 添加到 Notebook。

1. 在 Notebook 编辑器顶部菜单选择 `Add-ons` -> `Secrets`。
2. 新建标签为 `GOOGLE_API_KEY` 的 secret。
3. 把 API key 粘贴到 "Value" 字段并点击 "Save"。
4. 确认 `GOOGLE_API_KEY` 左侧复选框已勾选，使其附加到该 Notebook。

**3. 在 Notebook 中鉴权**

运行下面代码单元完成鉴权。

In [23]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Gemini API key setup complete.


### 1.3：导入 ADK 组件

现在导入 Agent Development Kit 与 Generative AI 的关键组件，保持代码结构清晰并确保能力齐全。

In [24]:
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.memory import InMemoryMemoryService
from google.adk.tools import load_memory, preload_memory
from google.genai import types

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


### 1.4：辅助函数

该辅助函数用于管理完整会话流程：创建/读取 session、处理 query、流式输出响应。

In [25]:
async def run_session(
    runner_instance: Runner, user_queries: list[str] | str, session_id: str = "default"
):
    """Helper function to run queries in a session and display responses."""
    print(f"\n### Session: {session_id}")

    # Create or retrieve session
    try:
        session = await session_service.create_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=session_id
        )
    except:
        session = await session_service.get_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=session_id
        )

    # Convert single query to list
    if isinstance(user_queries, str):
        user_queries = [user_queries]

    # Process each query
    for query in user_queries:
        print(f"\nUser > {query}")
        query_content = types.Content(role="user", parts=[types.Part(text=query)])

        # Stream agent response
        async for event in runner_instance.run_async(
            user_id=USER_ID, session_id=session.id, new_message=query_content
        ):
            if event.is_final_response() and event.content and event.content.parts:
                text = event.content.parts[0].text
                if text and text != "None":
                    print(f"Model: > {text}")


print("✅ Helper functions defined.")

✅ Helper functions defined.


### 1.5：配置重试选项

在使用 LLM 时，你可能会遇到速率限制或服务临时不可用等瞬时错误。重试选项会通过指数退避自动重试请求。

In [26]:
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

---
## 🤓 第 2 部分：Memory 工作流

通过前言你已经知道为什么需要 Memory。将 Memory 接入 Agent 的流程可分为**三步**：

**三步集成流程：**

1. **Initialize** -> 创建 `MemoryService` 并通过 `Runner` 提供给 Agent
2. **Ingest** -> 使用 `add_session_to_memory()` 将 session 数据写入 memory
3. **Retrieve** -> 使用 `search_memory()` 搜索已存 memories

下面逐步展开。

<img src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day3/memory-workflow.png" width="1400" alt="Memory workflow">

---
## 🧠 第 3 部分：初始化 MemoryService

### 3.1 初始化 Memory

ADK 通过 `BaseMemoryService` 接口提供多种 `MemoryService` 实现：

- **`InMemoryMemoryService`** - 适合原型与测试（关键词匹配，不持久化）
- **`VertexAiMemoryBankService`** - 托管云服务，支持 LLM consolidation 与语义检索
- **自定义实现** - 你可基于数据库自行实现，但生产更推荐托管服务

本 Notebook 使用 `InMemoryMemoryService` 学习核心机制。后续切换到 Vertex AI Memory Bank 这类生产方案时，调用方法保持一致。

In [27]:
memory_service = (
    InMemoryMemoryService()
)  # ADK's built-in Memory Service for development and testing

### 3.2 给 Agent 加入 Memory

接下来创建一个简单 Agent 用于回答用户问题。

In [28]:
# Define constants used throughout the notebook
APP_NAME = "MemoryDemoApp"
USER_ID = "demo_user"

# Create agent
user_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="MemoryDemoAgent",
    instruction="Answer user questions in simple words.",
)

print("✅ Agent created")

✅ Agent created


#### **创建 Runner**

现在把 Session 与 Memory 两个服务都提供给 `Runner`。

**关键配置：**

`Runner` 同时需要这两个服务才能启用完整 memory 能力：
- **`session_service`** -> 管理对话线程与 events
- **`memory_service`** -> 提供长期知识存储

二者协同方式是：Session 记录对话，Memory 负责跨 session 检索知识。

In [29]:
# Create Session Service
session_service = InMemorySessionService()  # Handles conversations

# Create runner with BOTH services
runner = Runner(
    agent=user_agent,
    app_name="MemoryDemoApp",
    session_service=session_service,
    memory_service=memory_service,  # Memory service is now available!
)

print("✅ Agent and Runner created with memory support!")

✅ Agent and Runner created with memory support!


### ‼️ 重要说明

**💡 配置不等于自动使用：** 给 `Runner` 配置 `memory_service` 只是让 memory 功能“可用”，不会自动触发。你仍需显式完成：
1. 用 `add_session_to_memory()` **写入数据**
2. 给 Agent 加上 memory 工具（`load_memory` 或 `preload_memory`）以**启用检索**

下面继续学习这些步骤。

### 3.3 MemoryService 实现选项

**本 Notebook：`InMemoryMemoryService`**
- 存储原始对话 events，不做 consolidation
- 基于关键词匹配
- 内存存储（重启后清空）
- 适合学习与本地开发

**生产环境：`VertexAiMemoryBankService`（Day 5 讲解）**
- 基于 LLM 提取关键事实
- 语义检索（按含义匹配）
- 持久化云存储
- 可集成外部知识源

**💡 API 一致性：** 两者都使用相同方法（`add_session_to_memory()`、`search_memory()`），因此你在这里学到的流程可直接迁移。

---
## 💾 第 4 部分：将 Session 数据写入 Memory

**为什么要把 Session 数据转入 Memory？**

memory 初始化后是空的，你需要把知识写进去。所有对话最初都存放在 Session 中，里面包含每条消息、工具调用与元数据。要让这些信息可用于长期回忆，需要显式调用 `add_session_to_memory()` 转入 memory。

这正是托管 memory 服务（如 Vertex AI Memory Bank）发挥价值的地方：**转入过程中会做智能 consolidation，提取关键事实并丢弃噪声。** 我们当前使用的 `InMemoryMemoryService` 不做 consolidation，只用于学习机制。

在转入之前需要先有数据。先与 Agent 对话，填充 session。该对话会像上一份 Notebook 一样保存在 SessionService 中。

In [30]:
# User tells agent about their favorite color
await run_session(
    runner,
    "My favorite color is blue-green. Can you write a Haiku about it?",
    "conversation-01",  # Session ID
)


### Session: conversation-01

User > My favorite color is blue-green. Can you write a Haiku about it?
Model: > A color of peace,
Calming like the ocean's waves,
Nature's gentle hue.


先验证对话已记录在 session 中。你应该能看到包含用户输入与模型响应的 session events。

In [31]:
session = await session_service.get_session(
    app_name=APP_NAME, user_id=USER_ID, session_id="conversation-01"
)

# Let's see what's in the session
print("📝 Session contains:")
for event in session.events:
    text = (
        event.content.parts[0].text[:60]
        if event.content and event.content.parts
        else "(empty)"
    )
    print(f"  {event.content.role}: {text}...")

📝 Session contains:
  user: My favorite color is blue-green. Can you write a Haiku about...
  model: A color of peace,
Calming like the ocean's waves,
Nature's g...


很好！session 已包含对话内容。现在可以调用 `add_session_to_memory()` 并传入 session 对象，将会话写入 memory 供后续搜索。

In [32]:
# This is the key method!
await memory_service.add_session_to_memory(session)

print("✅ Session added to memory!")

✅ Session added to memory!


---
## 🔎 第 5 部分：为 Agent 启用 Memory 检索

你已经把 session 数据写入 memory，但还差关键一步。**Agent 不能直接读取 MemoryService，需要通过工具进行检索。**

这是有意设计：让你精确控制“何时、如何”读取 memory。

### 5.1 ADK 的 Memory 检索方式

ADK 提供两个内置 memory 检索工具：

**`load_memory`（Reactive）**
- Agent 自主决定何时检索 memory
- 仅在“认为需要”时读取
- 更节省 token
- 风险：Agent 可能忘记检索

**`preload_memory`（Proactive）**
- 每轮对话前自动检索
- memory 始终可用
- 上下文更有保障，但效率更低
- 即使不需要也会检索

类比考试场景：`load_memory` 像“用到再查”，`preload_memory` 像“每次答题前都先复习全部笔记”。

### 5.2 给 Agent 加入 Load Memory 工具

先实现 reactive 方案。我们重建第 3 部分的 Agent，并把 `load_memory` 加入工具列表。由于这是 ADK 内置工具，无需自行实现，直接放入 tools 数组即可。

In [33]:
# Create agent
user_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="MemoryDemoAgent",
    instruction="Answer user questions in simple words. Use load_memory tool if you need to recall past conversations.",
    tools=[
        load_memory
    ],  # Agent now has access to Memory and can search it whenever it decides to!
)

print("✅ Agent with load_memory tool created.")

✅ Agent with load_memory tool created.


### 5.3 更新 Runner 并测试

现在把 Runner 切换到带 `load_memory` 的 `user_agent`，然后询问先前在其他 session 中存储过的“最喜欢的颜色”。

**👉 由于 session 之间不共享对话历史，Agent 若能正确回答，只能说明它调用了 `load_memory`，从长期 memory 中取回了信息。**

In [34]:
# Create a new runner with the updated agent
runner = Runner(
    agent=user_agent,
    app_name=APP_NAME,
    session_service=session_service,
    memory_service=memory_service,
)

await run_session(runner, "What is my favorite color?", "color-test")


### Session: color-test

User > What is my favorite color?


Model: > You told me your favorite color is blue-green.


### 5.4 完整手动流程测试

下面演示完整链路：先聊“生日”，手动写入 memory，再在新 session 中检索。完整流程即：**ingest -> store -> retrieve**。

In [35]:
await run_session(runner, "My birthday is on March 15th.", "birthday-session-01")


### Session: birthday-session-01

User > My birthday is on March 15th.
Model: > Okay, I will remember that your birthday is on March 15th.


现在手动把该 session 写入 memory。这一步很关键：它把对话从 session（短期）转入 memory（长期）。

In [36]:
# Manually save the session to memory
birthday_session = await session_service.get_session(
    app_name=APP_NAME, user_id=USER_ID, session_id="birthday-session-01"
)

await memory_service.add_session_to_memory(birthday_session)

print("✅ Birthday session saved to memory!")

✅ Birthday session saved to memory!


关键测试来了：我们用全新的 session ID 启动新会话，询问 Agent 能否回忆生日。

In [37]:
# Test retrieval in a NEW session
await run_session(
    runner, "When is my birthday?", "birthday-session-02"  # Different session ID
)


### Session: birthday-session-02

User > When is my birthday?


Model: > It looks like your birthday is on March 15th.


**发生了什么：**

1. Agent 收到问题：“When is my birthday?”
2. Agent 判断：需要历史上下文
3. Agent 调用：`load_memory("birthday")`
4. Memory 返回：包含 “March 15th” 的历史对话
5. Agent 回答：生日是 March 15th

即使是全新 session，也能通过 memory 检索成功回忆。

#### 🚀 轮到你：对比两种模式

把 tools 从 `load_memory` 改成 `preload_memory`（即 `tools=[preload_memory]`）后试试看。

**变化点：**
- `load_memory`（reactive）：由 Agent 判断是否检索
- `preload_memory`（proactive）：每轮自动预加载 memory

**可测试：**
1. 在新 session 里问 “What is my favorite color?”
2. 再问 “Tell me a joke” ，注意 `preload_memory` 即使不必要仍会检索
3. 思考不同场景该用哪种模式

### 5.5 手动搜索 Memory

除了让 Agent 通过工具检索，你也可以在代码里直接搜索 memory，适用于：
- 调试 memory 内容
- 构建分析面板  
- 开发自定义 memory 管理 UI

`search_memory()` 接收文本查询，返回 `SearchMemoryResponse`（包含命中的 memories）。

In [38]:
# Search for color preferences
search_response = await memory_service.search_memory(
    app_name=APP_NAME, user_id=USER_ID, query="What is the user's favorite color?"
)

print("🔍 Search Results:")
print(f"  Found {len(search_response.memories)} relevant memories")
print()

for memory in search_response.memories:
    if memory.content and memory.content.parts:
        text = memory.content.parts[0].text[:80]
        print(f"  [{memory.author}]: {text}...")

🔍 Search Results:
  Found 4 relevant memories

  [user]: My favorite color is blue-green. Can you write a Haiku about it?...
  [MemoryDemoAgent]: A color of peace,
Calming like the ocean's waves,
Nature's gentle hue....
  [user]: My birthday is on March 15th....
  [MemoryDemoAgent]: Okay, I will remember that your birthday is on March 15th....


#### **🚀 轮到你：测试不同查询**

尝试以下搜索，理解 `InMemoryMemoryService` 的关键词匹配行为：

1. **"what color does the user like"**
2. **"haiku"**
3. **"age"**
4. **"preferred hue"**

观察哪些查询有结果、哪些没有。你能发现什么规律？

**💡 关键理解：** memory 检索基于已存事实，Agent 不能凭空“幻觉”出不存在的记忆。

### 5.6 检索机制对比

**InMemoryMemoryService（本 Notebook）：**
- **方法**：关键词匹配
- **示例**：“favorite color” 能命中，因为原词存在
- **限制**：“preferred hue” 可能无法命中

**VertexAiMemoryBankService（Day 5）：**
- **方法**：基于 embedding 的语义检索
- **示例**：“preferred hue” 也能命中 “favorite color”
- **优势**：理解语义，而非仅匹配关键词

Day 5 会深入语义检索。

---
## 🤖 第 6 部分：自动化 Memory 存储

到目前为止，我们都是**手动**调用 `add_session_to_memory()`。生产系统通常需要这一步**自动完成**。

### 6.1 Callbacks

ADK 的 callback 机制可以让你在执行关键节点挂载逻辑。callback 是你定义并绑定到 Agent 的 **Python 函数**，ADK 会在特定阶段自动调用，就像执行流程里的检查点。

**可以把 callback 理解为 Agent 生命周期中的事件监听器。** Agent 处理请求会经历多个阶段：接收输入、调用 LLM、调用工具、生成响应。callback 让你在这些阶段插入自定义逻辑，而无需改动核心 Agent 代码。

**可用 callback 类型：**

- `before_agent_callback` -> Agent 开始处理前
- `after_agent_callback` -> Agent 完成当前轮后  
- `before_tool_callback` / `after_tool_callback` -> 工具调用前后
- `before_model_callback` / `after_model_callback` -> LLM 调用前后
- `on_model_error_callback` -> 发生错误时

**常见用途：**

- 日志与可观测性（记录 Agent 行为）
- 自动持久化数据（如自动写入 memory）
- 自定义校验或过滤
- 性能监控

**📚 参考：** [ADK Callbacks Documentation](https://google.github.io/adk-docs/agents/callbacks/)

![image.png](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day4/types_of_callbacks.png)

### 6.2 用 Callbacks 自动写入 Memory

为实现自动存储，我们使用 `after_agent_callback`。它会在每轮结束后触发，再调用 `add_session_to_memory()` 自动持久化当前对话。

但这里有个关键问题：callback 函数如何访问 memory service 与当前 session？答案是 `callback_context`。

当你定义 callback 函数时，ADK 会自动传入一个特殊参数 `callback_context`。它提供对 Memory Service 与其他运行时组件的访问能力。

**我们将这样用：** 在 callback 内读取当前 session 与 memory service，每轮结束后自动写入。

**💡 注意：** 你不需要手动创建这个 context，它由 ADK 在 callback 执行时自动注入。

In [39]:
async def auto_save_to_memory(callback_context):
    """Automatically save session to memory after each agent turn."""
    await callback_context._invocation_context.memory_service.add_session_to_memory(
        callback_context._invocation_context.session
    )


print("✅ Callback created.")

✅ Callback created.


### 6.3 创建 Agent：Callback + Preload Memory

现在创建一个同时具备以下能力的 Agent：
- **自动存储：** `after_agent_callback` 在每轮后写入 memory
- **自动检索：** `preload_memory` 在每轮前加载 memory

这样就得到一个“零手动干预”的自动 memory 系统。

In [40]:
# Agent with automatic memory saving
auto_memory_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="AutoMemoryAgent",
    instruction="Answer user questions.",
    tools=[preload_memory],
    after_agent_callback=auto_save_to_memory,  # Saves after each turn!
)

print("✅ Agent created with automatic memory saving!")

✅ Agent created with automatic memory saving!


**系统会自动发生：**

- 每轮响应后 -> callback 触发
- session 数据 -> 自动转入 memory
- 无需手动调用 `add_session_to_memory()`

框架会完成整套流程。

### 6.4 创建 Runner 并测试 Agent

现在开始测试：创建使用 auto-memory agent 的 Runner，并连接 session 与 memory 两个服务。

In [41]:
# Create a runner for the auto-save agent
# This connects our automated agent to the session and memory services
auto_runner = Runner(
    agent=auto_memory_agent,  # Use the agent with callback + preload_memory
    app_name=APP_NAME,
    session_service=session_service,  # Same services from Section 3
    memory_service=memory_service,
)

print("✅ Runner created.")

✅ Runner created.


In [42]:
# Test 1: Tell the agent about a gift (first conversation)
# The callback will automatically save this to memory when the turn completes
await run_session(
    auto_runner,
    "I gifted a new toy to my nephew on his 1st birthday!",
    "auto-save-test",
)

# Test 2: Ask about the gift in a NEW session (second conversation)
# The agent should retrieve the memory using preload_memory and answer correctly
await run_session(
    auto_runner,
    "What did I gift my nephew?",
    "auto-save-test-2",  # Different session ID - proves memory works across sessions!
)


### Session: auto-save-test

User > I gifted a new toy to my nephew on his 1st birthday!
Model: > That's wonderful! A first birthday is such a special milestone. I hope your nephew loves the toy!

### Session: auto-save-test-2

User > What did I gift my nephew?
Model: > You gifted your nephew a new toy.


**刚刚发生了什么：**

1. **第一次对话：** 提到了送给外甥的礼物
   - callback 自动写入 memory ✅
2. **第二次对话（新 session）：** 询问礼物内容  
   - `preload_memory` 自动检索 memory ✅
   - Agent 正确回答 ✅

**零手动 memory 调用！** 这就是自动 memory 管理的效果。

### 6.5 Session 写入 Memory 的频率建议

**常见策略：**

| 时机 | 实现方式 | 适用场景 |
|--------|----------------|----------|
| **每轮后写入** | `after_agent_callback` | 需要实时更新 memory |
| **会话结束后写入** | 会话结束时手动调用 | 批处理、减少 API 调用 |
| **定时写入** | 定时后台任务 | 长时运行对话 |

---
## 🧩 第 7 部分：Memory Consolidation

### 7.1 原始存储的局限

**到目前为止我们存了什么：**
- 每条用户消息
- 每条 Agent 响应  
- 每次工具调用

**问题在于：**
```
Session: 50 messages = 10,000 tokens
Memory:  All 50 messages stored
Search:  Returns all 50 messages -> Agent must process 10,000 tokens
```

这样不可扩展。我们需要 **consolidation**。

### 7.2 什么是 Memory Consolidation？

**Memory Consolidation** = 只提取**关键事实**，丢弃对话噪声。

**Before（原始存储）：**

```
User: "My favorite color is BlueGreen. I also like purple. 
       Actually, I prefer BlueGreen most of the time."
Agent: "Great! I'll remember that."
User: "Thanks!"
Agent: "You're welcome!"

-> Stores ALL 4 messages (redundant, verbose)
```

**After（consolidation）：**

```
Extracted Memory: "User's favorite color: BlueGreen"

-> Stores 1 concise fact
```

**收益：** 存储更省、检索更快、回答更准。

<img src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day3/memory-consolidation.png" width="1400" alt="Memory consolidation">

### 7.3 Consolidation 如何工作（概念）

**处理流程：**

```
1. Raw Session Events
   ↓
2. LLM analyzes conversation
   ↓
3. Extracts key facts
   ↓
4. Stores concise memories
   ↓
5. Merges with existing memories (deduplication)
```

**转换示例：**

```
Input:  "I'm allergic to peanuts. I can't eat anything with nuts."

Output: Memory {
  allergy: "peanuts, tree nuts"
  severity: "avoid completely"
}
```

自然语言会被提炼为结构化、可行动的数据。

### 7.4 Memory Consolidation 的下一步

**💡 关键点：** 托管 memory 服务会**自动**处理 consolidation。

**你调用的 API 不变：**
- `add_session_to_memory()` <- 同一个方法
- `search_memory()` <- 同一个方法

**差异在幕后实现：**
- **InMemoryMemoryService：** 存原始 events
- **VertexAiMemoryBankService：** 存储前先做智能 consolidation

**📚 参考：**
- [Vertex AI Memory Bank: Memory Consolidation Guide](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/memory-bank/generate-memories) -> Day 5 会实操。

---
## 📊 总结

你已经掌握了 ADK Memory 的**核心机制**：

1. **✅ 接入 Memory**
   - 与 `SessionService` 一起初始化 `MemoryService`
   - 两者共同提供给 `Runner`

2. **✅ 写入信息**
   - `await memory_service.add_session_to_memory(session)`
   - 将 session 数据转入长期存储
   - 可通过 callback 自动化

3. **✅ 检索 Memory**
   - `await memory_service.search_memory(app_name, user_id, query)`
   - 从历史对话中返回相关 memories

4. **✅ 在 Agent 中读取**
   - **Reactive：** `load_memory`（由 Agent 决定何时检索）
   - **Proactive：** `preload_memory`（每轮都提前加载到系统上下文）

5. **✅ Memory Consolidation**
   - 从 Session 数据中提炼关键信息
   - 由 Vertex AI Memory Bank 等托管服务提供

## 🎉 **恭喜！** 你已掌握 ADK Memory 管理

**📚 继续学习：**
- [ADK Memory Documentation](https://google.github.io/adk-docs/sessions/memory/)
- [Vertex AI Memory Bank](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/memory-bank/overview)
- [Memory Consolidation Guide](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/memory-bank/generate-memories)

**🎯 下一步：**

准备进入 Day 4 吗？学习如何**实现 Observability 并评估 Agent**，确保其在生产环境中按预期工作。